<style>
table {
  margin-left: 0 !important;
  margin-right: auto !important;
}
th, td {
  text-align: left !important;
}
</style>

## Systems Thinking · From a Real Problem to an Optimized Decision

**How much cooling should the classroom use, and what happens when that decision changes?**

Systems thinking turns this real question into a simplified system whose behavior and performance can be evaluated.

The complete reasoning chain is:

> **1. Frame the problem → 2. Select elements → 3. Build the system equation → 4. Simulate behavior → 5. Evaluate performance → 6. Select a preferred feasible decision**

### 1 · Frame the real problem

“The room is uncomfortable” is a symptom. A system model begins by defining the purpose and boundary: keep one classroom comfortable over the period of interest while accounting for energy use. A solution is not chosen yet.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="assets/01_real_problem_labeled.png" alt="Labeled classroom problem showing discomfort, weather, cooling, and indoor temperature" width="570" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

The picture separates the observed symptom from possible influences: weather, occupants, openings, cooling, indoor temperature, energy, and time.

### 2 · Select the elements that matter

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="assets/02_candidate_elements_labeled.png" alt="Labeled candidate classroom elements including temperatures, occupants, openings, cooling, energy, and time" width="570" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

The icons are candidates, not yet a system. The simplified model keeps outdoor temperature, occupants, indoor temperature, cooling, energy, and time. Door and window effects are omitted in this first model. This is a modeling choice: include what is needed for the decision, then revise the boundary if the model is insufficient.


### 3 · Assign roles and build the system equation

Each selected element receives a role. The role determines whether the element enters the system as a given value, an uncontrolled input, an evolving state, a decision, an output, a limit, or an analyst setting.

| Role | Question answered |
|:---|:---|
| Fixed parameter | What is treated as given? |
| External input | What changes but is not controlled? |
| System state | What condition carries forward over time? |
| Decision variable | What action can be chosen? |
| Performance output | What result is measured? |
| Constraint | What must every acceptable candidate satisfy? |
| Hyperparameter | What analyst setting changes evaluation or search? |

Changing is not the same as choosing. A decision acts on the real system; a hyperparameter acts on the analysis.

The roles define three connected mappings:

<div style="height: 0.4rem;"></div>

> $\displaystyle \text{Next state}=F(\text{current state},\text{external input},\text{decision};\text{fixed parameters})$
>
> $\displaystyle \text{Performance}=G(\text{state path},\text{decision};\text{fixed parameters})$
>
> $\displaystyle \text{Score}=H(\text{performance};\text{hyperparameter})$

<div style="height: 0.65rem;"></div>

\(F\) describes system behavior, \(G\) measures the resulting performance, and \(H\) evaluates that performance. The semicolon separates values treated as fixed during one analysis.

The classroom model now has a clear structure:

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="assets/03_simplified_system_labeled.png" alt="Labeled system diagram mapping outdoor temperature, occupancy, cooling decision, fixed effects, and indoor state variables" width="570" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

The diagrams use \(T\) and \(u\) as shorthand for a complete state path and decision. A subscript \(t\) always refers to one time step; the diagram labels \(T_{\mathrm{out}}\) and \(T_{\mathrm{next}}\) represent \(T_t^{\mathrm{out}}\) and \(T_{t+1}\), respectively.

Outdoor temperature and occupants add heat, while cooling removes heat. This relationship becomes the state equation:

<div style="height: 0.4rem;"></div>

> $\displaystyle T_{t+1}=T_t+a\left(T_t^{\mathrm{out}}-T_t\right)+bN_t-cu_t$

<div style="height: 0.65rem;"></div>

| Variable | Meaning | Role |
|:---|:---|:---|
| $T_t$ | Current indoor temperature | System state |
| $T_t^{\mathrm{out}}$ | Outdoor temperature | External input |
| $N_t$ | Number of occupants | External input |
| $u_t$ | Cooling level | Decision variable |
| $a,b,c$ | Weather, occupant-heat, and cooling effects | Fixed parameters |

The horizon contains \(n=12\) decision steps. Thus \(t=0,\ldots,n-1\), the state path is \(T_0,\ldots,T_n\), and \(u_t\) is the cooling action at step \(t\).

For the demonstrations, the complete decision is represented by two values:

> $\displaystyle u=(u_{\mathrm{early}},u_{\mathrm{late}}),\qquad
u_t=
\begin{cases}
u_{\mathrm{early}},&t=0,\ldots,5,\\
u_{\mathrm{late}},&t=6,\ldots,11.
\end{cases}$

For the controlled demonstrations, \(T_0=27\,^\circ\mathrm{C}\), \(T_t^{\mathrm{out}}=31\,^\circ\mathrm{C}\), \(N_t=20\), and \((a,b,c)=(0.12,0.012,0.45)\). These quantities stay fixed while the cooling decision changes.

At each time step, the model starts with \(T_t\), adds weather and occupant effects, subtracts the cooling effect, and produces \(T_{t+1}\). Repeating this update produces the complete temperature path.

The energy weight does not appear in the state equation. It cannot change the temperature path for a fixed cooling decision; it is introduced later when alternatives are compared.

### 4 · Simulate the system and visualize its behavior

The next cell runs a controlled comparison under the fixed conditions above.


In [ ]:
# Canonical classroom system used throughout Lectures 01-1, 01-2, and 02-1.
import numpy as np

TIME_STEPS = 12
INITIAL_TEMPERATURE = 27.0

# Fixed parameters
WEATHER_EXCHANGE = 0.12
OCCUPANT_HEAT = 0.012
COOLING_EFFECT = 0.45

# External inputs
OUTSIDE_TEMPERATURE = np.full(TIME_STEPS, 31.0)
OCCUPANTS = np.full(TIME_STEPS, 20.0)

# Requirement limits
MIN_COOLING = 0.0
MAX_COOLING = 5.0
MIN_TEMPERATURE = 20.0
MAX_TEMPERATURE = 30.0
MAX_ENERGY = 60.0


def expand_decision(decision):
    """Expand (early cooling, late cooling) into the 12-step decision schedule."""
    early_cooling, late_cooling = map(float, decision)
    return np.r_[
        np.full(6, early_cooling),
        np.full(6, late_cooling),
    ]


def simulate_classroom(decision):
    """Apply the 01-1 state equation and return the complete state path."""
    cooling_schedule = expand_decision(decision)
    temperatures = [INITIAL_TEMPERATURE]

    for outdoor, people, cooling in zip(
        OUTSIDE_TEMPERATURE, OCCUPANTS, cooling_schedule
    ):
        current = temperatures[-1]
        next_temperature = (
            current
            + WEATHER_EXCHANGE * (outdoor - current)
            + OCCUPANT_HEAT * people
            - COOLING_EFFECT * cooling
        )
        temperatures.append(next_temperature)

    return cooling_schedule, np.asarray(temperatures)


def performance_outputs(cooling_schedule, temperatures):
    """Calculate the raw performance outputs D and E defined in 01-1."""
    discomfort = np.sum(
        np.maximum(temperatures[1:] - 24.0, 0.0) ** 2
        + np.maximum(22.0 - temperatures[1:], 0.0) ** 2
    )
    energy = 0.5 * np.sum(cooling_schedule ** 2)
    return float(discomfort), float(energy)


def evaluate_candidate(decision, energy_weight=1.0):
    """Simulate one decision, check requirements, and calculate its score."""
    decision = tuple(map(float, decision))
    cooling_schedule, temperatures = simulate_classroom(decision)
    discomfort, energy = performance_outputs(cooling_schedule, temperatures)

    violations = []
    if not all(MIN_COOLING <= value <= MAX_COOLING for value in decision):
        violations.append("cooling bound")
    if temperatures[1:].min() < MIN_TEMPERATURE:
        violations.append("minimum temperature")
    if temperatures[1:].max() > MAX_TEMPERATURE:
        violations.append("maximum temperature")
    if energy > MAX_ENERGY:
        violations.append("energy limit")

    return {
        "decision": decision,
        "cooling_schedule": cooling_schedule,
        "temperatures": temperatures,
        "discomfort": discomfort,
        "energy": energy,
        "energy_weight": float(energy_weight),
        "objective": discomfort + float(energy_weight) * energy,
        "feasible": not violations,
        "violations": tuple(violations),
    }


def enumerate_grid_candidates(energy_weight=1.0, step=0.5):
    """Evaluate the transparent decision grid used in Lectures 01-1 and 01-2."""
    levels = np.arange(MIN_COOLING, MAX_COOLING + step / 2, step)
    return [
        evaluate_candidate((early, late), energy_weight)
        for early in levels
        for late in levels
    ]


def best_grid_candidate(energy_weight=1.0, step=0.5):
    """Return the lowest-scoring feasible candidate on the stated grid."""
    candidates = enumerate_grid_candidates(energy_weight, step)
    return min(
        (candidate for candidate in candidates if candidate["feasible"]),
        key=lambda candidate: candidate["objective"],
    )


import sys
from types import SimpleNamespace

import matplotlib

def _pyplot(*, interactive=False):
    """Return pyplot, activating ipympl for interactive figures when available."""
    if interactive and sys.platform != "emscripten":
        try:
            matplotlib.use("widget", force=True)
        except (RuntimeError, ValueError):
            from matplotlib.backends import backend_registry

            backend_registry._clear()
            matplotlib.use("widget", force=True)
    import matplotlib.pyplot as plt

    return plt

def show_system_behavior(evaluate_candidate, *, time_steps=12):
    """Plot a controlled comparison of three constant cooling decisions."""
    plt = _pyplot()
    decisions = ((1.0, 1.0), (3.0, 3.0), (4.0, 4.0))
    labels = ("Low cooling", "Moderate cooling", "Strong cooling")
    colors = ("tab:red", "tab:blue", "tab:purple")
    results = [evaluate_candidate(decision) for decision in decisions]

    decision_steps = np.arange(time_steps)
    state_steps = np.arange(time_steps + 1)
    figure, axes = plt.subplots(1, 2, figsize=(10.8, 3.8))

    for label, color, result in zip(labels, colors, results):
        axes[0].step(
            decision_steps,
            result["cooling_schedule"],
            where="mid",
            linewidth=2,
            color=color,
            label=label,
        )
    axes[0].set(
        xlabel="Time step",
        ylabel="Cooling level",
        title="Candidate decisions",
        xlim=(0, time_steps - 1),
        ylim=(0, 5),
    )
    axes[0].legend(fontsize=8)

    axes[1].axhspan(22, 24, color="tab:green", alpha=0.15, label="Comfort range")
    axes[1].plot(
        state_steps,
        np.full_like(state_steps, 31.0, dtype=float),
        linestyle="--",
        color="black",
        linewidth=1.5,
        label="Outdoor temperature: fixed at 31 °C",
    )
    for label, color, result in zip(labels, colors, results):
        axes[1].plot(
            state_steps,
            result["temperatures"],
            marker="o",
            linewidth=2,
            color=color,
            label=label,
        )
    axes[1].set(
        xlabel="Time step",
        ylabel="Indoor temperature (°C)",
        title="Different decisions produce different behavior",
        xlim=(0, time_steps),
        ylim=(18, 32),
    )
    axes[1].legend(fontsize=8)

    for axis in axes:
        axis.grid(alpha=0.25)
    figure.suptitle(
        "Controlled comparison · same outdoor temperature and occupancy",
        fontsize=11,
    )
    figure.tight_layout(rect=(0, 0, 1, 0.94))
    plt.show()
    plt.close(figure)
    return figure

system_behavior_figure = show_system_behavior(
    evaluate_candidate, time_steps=TIME_STEPS
)

The code above compares three constant cooling decisions over \(n=12\) time steps: \(u=(1,1)\), \(u=(3,3)\), and \(u=(4,4)\). Outdoor temperature remains at 31 °C and occupancy remains at 20 people for every candidate. Only the cooling decision changes, so differences in the result can be attributed to that decision.

- The **left graph** shows low, moderate, and strong cooling schedules under the same external conditions.
- The **right graph** shows the temperature path produced by each schedule. Low cooling leaves the room hot, moderate cooling moves it into the comfort range, and strong cooling eventually makes it too cold.

The paths separate even though outdoor temperature and occupancy are identical. The model therefore makes the causal chain visible: **decision → system behavior → performance**.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="assets/04_system_performance_labeled.png" alt="Labeled candidate-performance diagram connecting decision, state path, discomfort, energy, and feasibility" width="570" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

### 5 · Measure performance and test feasibility

The temperature path and cooling schedule are converted into two raw performance measures:

<div style="height: 0.4rem;"></div>

> $\displaystyle D(u)=\operatorname{Discomfort}\!\left(T_1(u),\ldots,T_n(u)\right)$
>
> $\displaystyle E(u)=\operatorname{Energy}\!\left(u_0,\ldots,u_{n-1}\right)$

<div style="height: 0.65rem;"></div>

Discomfort \(D\) increases when the temperature leaves the 22–24 °C comfort range. Energy \(E\) increases with cooling effort. These values describe the candidate before any analyst preference is applied.

A candidate is feasible only when every requirement holds:

| Requirement | Symbolic limit | Value |
|:---|:---|:---|
| Early and late cooling | $u_{\min}\le u_{\mathrm{early}},u_{\mathrm{late}}\le u_{\max}$ | $u_{\min}=0,\ u_{\max}=5$ |
| Indoor temperature | $T_{\min}\le T_t\le T_{\max}$, $t=1,\ldots,n$ | $T_{\min}=20\,^\circ\mathrm{C},\ T_{\max}=30\,^\circ\mathrm{C}$ |
| Total energy | $E(u)\le E_{\max}$ | $E_{\max}=60$ |

The evaluation sequence for one candidate is therefore:

> **Choose \(u\) → simulate \(T\) → calculate \(D\) and \(E\) → check the constraints**

An infeasible candidate is rejected, regardless of its score.

### 6 · Compare feasible candidates and select a preferred decision

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="assets/05_system_optimization_wide_labeled.png" alt="Labeled optimization diagram connecting decision, system state, raw performance, energy weight, score, and preferred candidate" width="570" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

The objective combines discomfort and energy for each feasible decision:

<div style="height: 0.4rem;"></div>

> $\displaystyle J(u;\lambda_E)=H\!\left(D(u),E(u);\lambda_E\right)=D(u)+\lambda_EE(u)$
>
> $\displaystyle \text{Preferred decision}=\text{feasible }u\text{ with the lowest }J$

<div style="height: 0.65rem;"></div>

Here, \(u\) is the decision and \(\lambda_E\) is the energy-weight hyperparameter. Changing \(u\) changes the physical system; changing \(\lambda_E\) changes only how the same performance is scored.

The optimization follows a transparent search:

1. Generate candidate pairs of early and late cooling levels on a 0.5-unit grid.
2. Simulate the temperature path for each pair.
3. Calculate \(D\) and \(E\), then reject infeasible candidates.
4. Calculate \(J\) for every feasible candidate.
5. Select the feasible grid candidate with the lowest \(J\).

The grid keeps the search visible. Its star marks the best sampled candidate, not a guaranteed optimum over every possible continuous value.

The interactive figure makes the complete chain visible:

- **Decision change:** the temperature path in Panel 1 and the square in Panels 2–3 move together. Changes in \(D\), \(E\), and feasibility show the physical consequence of that decision.
- **Current candidate versus search result:** the square represents the slider-selected candidate; the star represents the lowest-scoring feasible grid candidate. When they do not coincide, the current choice is not the grid benchmark.
- **Hyperparameter change with the decision fixed:** the temperature path, square, \(D\), and \(E\) stay unchanged. The score landscape and star can change because \(\lambda_E\) changes the comparison rule, not the physical system.

The result is not merely a final number. It is a traceable journey from a real problem, through a system model and measurable performance, to a preferred feasible decision under a stated evaluation rule.


In [ ]:
def show_classroom_explorer(
    evaluate_candidate,
    enumerate_candidates,
    *,
    time_steps=12,
    min_cooling=0.0,
    max_cooling=5.0,
    min_temperature=20.0,
    max_temperature=30.0,
    initial_decision=(3.0, 2.0),
):
    """Build the shared decision/state/performance interactive explorer."""
    plt = _pyplot(interactive=True)
    from matplotlib.widgets import Slider

    plt.close("all")
    step = 0.5
    baseline = enumerate_candidates(energy_weight=1.0, step=step)
    levels = np.arange(min_cooling, max_cooling + step / 2, step)
    feasible_points = np.array([
        (record["energy"], record["discomfort"])
        for record in baseline if record["feasible"]
    ])
    infeasible_points = np.array([
        (record["energy"], record["discomfort"])
        for record in baseline if not record["feasible"]
    ])

    def objective_landscape(energy_weight):
        values = np.full((len(levels), len(levels)), np.nan)
        records = enumerate_candidates(energy_weight, step=step)
        for record in records:
            early, late = record["decision"]
            if record["feasible"]:
                values[
                    int(round((early - min_cooling) / step)),
                    int(round((late - min_cooling) / step)),
                ] = record["objective"]
        best = min(
            (record for record in records if record["feasible"]),
            key=lambda record: record["objective"],
        )
        return values, best

    initial_weight = 1.0
    current = evaluate_candidate(initial_decision, initial_weight)
    initial_landscape, best = objective_landscape(initial_weight)
    figure, axes = plt.subplots(1, 3, figsize=(12, 8))
    figure.subplots_adjust(left=0.07, right=0.98, bottom=0.35, top=0.80, wspace=0.34)

    axes[0].axhspan(
        min_temperature,
        max_temperature,
        color="tab:blue",
        alpha=0.07,
        label="Allowed range",
    )
    axes[0].axhspan(22, 24, color="tab:green", alpha=0.18, label="Comfort target")
    temperature_line, = axes[0].plot(
        np.arange(time_steps + 1),
        current["temperatures"],
        marker="o",
        linewidth=2,
        color="tab:blue",
    )
    axes[0].set(
        xlabel="Time step",
        ylabel="Indoor temperature (°C)",
        xlim=(0, time_steps),
        ylim=(17, 34),
    )
    axes[0].grid(alpha=0.25)
    axes[0].legend(fontsize=8)

    axes[1].scatter(
        infeasible_points[:, 0],
        infeasible_points[:, 1],
        marker="x",
        color="lightgray",
        label="Infeasible grid candidate",
    )
    axes[1].scatter(
        feasible_points[:, 0],
        feasible_points[:, 1],
        color="teal",
        alpha=0.55,
        label="Feasible grid candidate",
    )
    current_performance = axes[1].scatter(
        current["energy"], current["discomfort"],
        marker="s", s=90, color="tab:orange", edgecolor="black",
        label="Current candidate",
    )
    best_performance = axes[1].scatter(
        best["energy"], best["discomfort"],
        marker="*", s=190, color="gold", edgecolor="black",
        label="Best grid candidate",
    )
    all_points = feasible_points.tolist() + infeasible_points.tolist()
    axes[1].set(
        xlabel="Raw energy performance $E$",
        ylabel="Raw discomfort performance $D$",
        xlim=(0, max(point[0] for point in all_points) * 1.07),
        ylim=(0, max(point[1] for point in all_points) * 1.07),
        title="Raw performance and feasibility",
    )
    axes[1].grid(alpha=0.25)
    axes[1].legend(fontsize=7)

    color_map = plt.colormaps["viridis_r"].copy()
    color_map.set_bad("#e6e6e6")
    objective_image = axes[2].imshow(
        initial_landscape,
        origin="lower",
        cmap=color_map,
        extent=(min_cooling - 0.25, max_cooling + 0.25) * 2,
        aspect="equal",
    )
    current_decision = axes[2].scatter(
        initial_decision[1], initial_decision[0],
        marker="s", s=90, color="tab:orange", edgecolor="black",
        label="Current candidate",
    )
    best_decision = axes[2].scatter(
        best["decision"][1], best["decision"][0],
        marker="*", s=190, color="gold", edgecolor="black",
        label="Best grid candidate",
    )
    axes[2].set(
        xticks=np.arange(min_cooling, max_cooling + 0.1, 1),
        yticks=np.arange(min_cooling, max_cooling + 0.1, 1),
        xlabel="Late cooling decision",
        ylabel="Early cooling decision",
    )
    axes[2].legend(fontsize=8)

    status_text = figure.text(
        0.5, 0.955, "", ha="center", va="top", fontsize=12, fontweight="bold"
    )
    metrics_text = figure.text(0.5, 0.915, "", ha="center", va="top", fontsize=10)
    figure.text(
        0.5,
        0.255,
        "Fixed system model: a=0.12, b=0.012, c=0.45; "
        "external inputs: outdoor temperature=31 °C, occupants=20",
        ha="center",
        va="center",
        fontsize=9,
        bbox=dict(boxstyle="round,pad=0.35", facecolor="#f1f3f5", edgecolor="#adb5bd"),
    )

    early_axis = figure.add_axes([0.35, 0.175, 0.55, 0.025])
    late_axis = figure.add_axes([0.35, 0.115, 0.55, 0.025])
    weight_axis = figure.add_axes([0.35, 0.055, 0.55, 0.025])
    early_slider = Slider(
        early_axis,
        "Decision · early cooling",
        min_cooling,
        max_cooling,
        valinit=initial_decision[0],
        valstep=0.25,
        valfmt="%1.2f",
        color="tab:blue",
    )
    late_slider = Slider(
        late_axis,
        "Decision · late cooling",
        min_cooling,
        max_cooling,
        valinit=initial_decision[1],
        valstep=0.25,
        valfmt="%1.2f",
        color="tab:blue",
    )
    weight_slider = Slider(
        weight_axis,
        "Hyperparameter · $\\lambda_E$",
        0.0,
        3.0,
        valinit=initial_weight,
        valstep=0.1,
        valfmt="%1.1f",
        color="tab:purple",
    )
    state = {}

    def refresh(_=None):
        decision = (early_slider.val, late_slider.val)
        energy_weight = weight_slider.val
        current = evaluate_candidate(decision, energy_weight)
        landscape, best = objective_landscape(energy_weight)
        status = "FEASIBLE" if current["feasible"] else "INFEASIBLE"
        temperature_line.set_ydata(current["temperatures"])
        current_performance.set_offsets([[current["energy"], current["discomfort"]]])
        best_performance.set_offsets([[best["energy"], best["discomfort"]]])
        objective_image.set_data(landscape)
        objective_image.set_clim(np.nanmin(landscape), np.nanmax(landscape))
        current_decision.set_offsets([[decision[1], decision[0]]])
        best_decision.set_offsets([[best["decision"][1], best["decision"][0]]])
        axes[0].set_title(f"State path: {status.lower()}")
        axes[2].set_title(f"Objective landscape\n$\\lambda_E$={energy_weight:.1f}")
        status_text.set_text(
            f"Current decision u=({decision[0]:.2f}, {decision[1]:.2f}) · {status}"
        )
        status_text.set_color("#087f5b" if current["feasible"] else "#c92a2a")
        eligibility = "eligible" if current["feasible"] else "rejected before comparison"
        metrics_text.set_text(
            f"D={current['discomfort']:.2f}, E={current['energy']:.2f}, "
            f"J=D+{energy_weight:.1f}E={current['objective']:.2f} · {eligibility}  |  "
            f"best grid u=({best['decision'][0]:.1f}, {best['decision'][1]:.1f})"
        )
        state.clear()
        state.update(current=current, best=best, energy_weight=energy_weight)
        figure.canvas.draw_idle()

    for slider in (early_slider, late_slider, weight_slider):
        slider.on_changed(refresh)
    figure._classroom_sliders = (early_slider, late_slider, weight_slider)
    refresh()
    plt.show()
    return SimpleNamespace(
        figure=figure,
        state=state,
        refresh=refresh,
        early_slider=early_slider,
        late_slider=late_slider,
        energy_weight_slider=weight_slider,
    )

explorer = show_classroom_explorer(
    evaluate_candidate,
    enumerate_grid_candidates,
    time_steps=TIME_STEPS,
    min_cooling=MIN_COOLING,
    max_cooling=MAX_COOLING,
    min_temperature=MIN_TEMPERATURE,
    max_temperature=MAX_TEMPERATURE,
    initial_decision=(3.5, 1.5),
)
systems_thinking_state = explorer.state

### Takeaway

Systems thinking turns a vague concern into a traceable evaluation chain:

> **set the boundary → assign roles → predict states with \(F\) → measure performance with \(G\) → check requirements → score feasible candidates with \(H\)**

For the classroom example, the decision \(u=(u_{\mathrm{early}},u_{\mathrm{late}})\) expands into \(u_0,\ldots,u_{n-1}\), the state equation produces \(T_1,\ldots,T_n\), and the raw outputs \(D(u),E(u)\) support feasibility checks and the score \(J(u;\lambda_E)\).

Lecture 01-2 keeps this system model and evaluation chain fixed. It next makes the decision variables, objective, and constraints explicit.
